In [5]:
"""
generate_data.py
================
Erzeugt synthetische Messdaten (Kraft + Beschleunigung) für die
Bremsenquietsch-Analyse und speichert sie als CSV-Dateien.

Für jeden Bremsdruck 1..20 bar wird eine Datei erzeugt:
    data/measurement_<druck>bar.csv

Spalten:
    time_s      – Zeit in Sekunden
    force_N     – externe Kraft in Newton
    accel_m_s2  – Beschleunigung in m/s²
"""

import os
import numpy as np

# ----- Parameter (an die Masterarbeit angelehnt) -----
FS = 44100.0          # Abtastrate [Hz] Datei ist zu größ deswegen 
DURATION = 1.0        # Messdauer [s] (kürzer als 31 s für handliche Dateien)
SWEEP_F0 = 200.0      # Startfrequenz [Hz]
SWEEP_F1 = 3000.0     # Endfrequenz [Hz]
SWEEP_T = 0.5        # Sweepdauer [s]
F_CRIT = 2600.0       # kritische Frequenz [Hz]

OUTPUT_DIR = "data"


def linear_sweep(t, f0, f1, duration):
    phase = 2.0 * np.pi * (f0 * t + 0.5 * (f1 - f0) / duration * t ** 2)
    return np.sin(phase)


def make_sweep_train(total_time, fs):
    n_total = int(total_time * fs)
    n_sweep = int(SWEEP_T * fs)
    t_sweep = np.arange(n_sweep) / fs
    single = linear_sweep(t_sweep, SWEEP_F0, SWEEP_F1, SWEEP_T)
    n_repeats = n_total // n_sweep + 1
    train = np.tile(single, n_repeats)[:n_total]
    fade = int(0.01 * fs)
    env = np.ones(n_sweep)
    env[:fade] = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    env_full = np.tile(env, n_repeats)[:n_total]
    return train * env_full


def simulate(pressure_bar, seed):
    rng = np.random.default_rng(seed)
    t = np.arange(int(DURATION * FS)) / FS
    f_ext = make_sweep_train(DURATION, FS)

    # Übertragungsfunktion: Grundsystem + Resonanzpeak bei f_crit
    omega = 2.0 * np.pi * np.fft.rfftfreq(len(t), d=1.0 / FS)

    base_omega0 = 2.0 * np.pi * 1500.0
    D = 0.05
    H_base = 1.0 / (base_omega0 ** 2 - omega ** 2 + 2j * D * base_omega0 * omega)

    # Peak wächst mit Bremsdruck (ab 6 bar deutlich)
    peak_gain = 0.3 + 0.05 * pressure_bar * (pressure_bar > 5)
    peak_width = 80.0
    peak = peak_gain / (1.0 + ((omega - 2 * np.pi * F_CRIT) / (2 * np.pi * peak_width)) ** 2)
    peak_complex = peak * np.exp(-1j * np.pi / 2)

    H = H_base + peak_complex

    F = np.fft.rfft(f_ext)
    A = H * F
    acc = np.fft.irfft(A, n=len(t))

    # Rauschen
    acc += 0.01 * rng.standard_normal(len(t))
    f_ext += 0.005 * rng.standard_normal(len(t))

    return t, f_ext, acc


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for p in range(1, 11):
        t, f, a = simulate(pressure_bar=p, seed=p)
        filename = os.path.join(OUTPUT_DIR, f"measurement_{p:02d}bar.csv")

        # CSV schreiben: Zeit, Kraft, Beschleunigung
        data = np.column_stack([t, f, a])
        header = "time_s,force_N,accel_m_s2"
        np.savetxt(filename, data, delimiter=",", header=header, comments="", fmt="%.4g")
        print(f"  geschrieben: {filename}  ({len(t)} Zeilen)")

    print("\nFertig! Alle CSV-Dateien liegen im Ordner:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

  geschrieben: data/measurement_01bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_02bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_03bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_04bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_05bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_06bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_07bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_08bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_09bar.csv  (44100 Zeilen)
  geschrieben: data/measurement_10bar.csv  (44100 Zeilen)

Fertig! Alle CSV-Dateien liegen im Ordner: data


In [6]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

FS_DEFAULT = 10000.0     # muss zum Generator passen
N_BLOCKS = 5
F_MIN, F_MAX = 200.0, 4000.0


def hann_window(n):
    return 0.5 * (1.0 - np.cos(2.0 * np.pi * np.arange(n) / n))


def block_fft(signal, fs, n_blocks):
    block_size = len(signal) // n_blocks
    blocks = signal[: n_blocks * block_size].reshape(n_blocks, block_size)
    window = hann_window(block_size)
    wc = 2.0 / window.sum()
    spectra = np.array([np.fft.rfft(b * window) * wc for b in blocks])
    return spectra.mean(axis=0), np.fft.rfftfreq(block_size, d=1.0 / fs)


def analyze_file(path):
    data = np.loadtxt(path, delimiter=",", skiprows=1)
    t, f, a = data[:, 0], data[:, 1], data[:, 2]

    # Abtastrate aus Zeitachse schätzen
    fs = 1.0 / np.mean(np.diff(t))

    F, freqs = block_fft(f, fs, N_BLOCKS)
    A, _ = block_fft(a, fs, N_BLOCKS)

    eps = 1e-12 * np.max(np.abs(F))
    H = np.where(np.abs(F) > eps, A / F, 0.0 + 0.0j)

    omega = 2.0 * np.pi * freqs
    omega[0] = np.inf  # DC vermeiden

    W_ext = (np.pi / omega ** 2) * np.abs(H) * np.sin(np.angle(H))
    W_ext = np.nan_to_num(W_ext, nan=0.0)
    return freqs, W_ext, F, A, H


def main():
    files = sorted(glob.glob("data/measurement_*bar.csv"))
    if not files:
        print("Keine Dateien in data/ gefunden.")
        return

    results = {}
    spectra_demo = None

    for path in files:
        pressure = int(os.path.basename(path).split("_")[1].replace("bar.csv", ""))
        freqs, W, F, A, H = analyze_file(path)
        results[pressure] = (freqs, W)
        if pressure == 5:
            spectra_demo = (freqs, F, A, H)

    # ---- Plot 1: Spektren bei 5 bar ----
    freqs, F, A, H = spectra_demo
    mask = (freqs >= F_MIN) & (freqs <= F_MAX)
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    eps = 1e-20
    axes[0].plot(freqs[mask], 20 * np.log10(np.abs(F[mask]) + eps))
    axes[0].set_ylabel("Kraft [dB]")
    axes[0].set_title("Gemitteltes Kraftspektrum")
    axes[1].plot(freqs[mask], 20 * np.log10(np.abs(A[mask]) + eps))
    axes[1].set_ylabel("Beschleunigung [dB]")
    axes[1].set_title("Gemitteltes Beschleunigungsspektrum")
    axes[2].plot(freqs[mask], 20 * np.log10(np.abs(H[mask]) + eps))
    axes[2].set_ylabel("|H| [dB]")
    axes[2].set_xlabel("Frequenz [Hz]")
    axes[2].set_title("Übertragungsspektrum")
    fig.tight_layout()
    fig.savefig("figure_spectra.png", dpi=120)
    plt.close(fig)
    print("gespeichert: figure_spectra.png")

    # ---- Plot 2: 2D-Karte ----
    pressures = sorted(results.keys())
    freqs_common = results[pressures[0]][0]
    mask = (freqs_common >= F_MIN) & (freqs_common <= F_MAX)
    f_plot = freqs_common[mask]
    W_matrix = np.vstack([results[p][1][mask] for p in pressures])
    p_grid = np.repeat(np.array(pressures)[:, None], len(f_plot), axis=1)
    f_grid = np.repeat(f_plot[None, :], len(pressures), axis=0)

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.pcolormesh(f_grid, p_grid, W_matrix, shading="auto", cmap="jet")
    ax.set_xlabel("Frequenz [Hz]")
    ax.set_ylabel("Bremsdruck [bar]")
    ax.set_title(r"Normierte externe Arbeit $\tilde{W}_{ext}$")
    fig.colorbar(im, ax=ax, label=r"$\tilde{W}_{ext}$ [m/N]")
    fig.tight_layout()
    fig.savefig("figure_work_map.png", dpi=120)
    plt.close(fig)
    print("gespeichert: figure_work_map.png")


if __name__ == "__main__":
    main()

gespeichert: figure_spectra.png
gespeichert: figure_work_map.png
